In [1]:
import gym
import energym

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import time

from stable_baselines3 import DQN

import random
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [2]:
# Parameters:
years = 7

frequency = 4   # DA on every hour

gamma = 0.9
learning_starts = 35040/2   # training starts 6 months in
target_update_interval = 5000
buffer_size = 1000000
exploration_fraction = 0.8
exploration_initial_eps = 1
exploration_final_eps = 0.05
total_timesteps = years * 35040   # 7 years for training
eval_freq = 35040/2

In [3]:
!pip install tqdm

In [4]:
!pip install 'shimmy>=0.2.1'

# NO DA

## Training no DA

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from tqdm.notebook import tqdm

class ProgressBarCallback(BaseCallback):
    """
    Display a progress bar when training SB3 agent
    """
    def __init__(self, total_timesteps):
        super().__init__()
        self.pbar = None
        self.total_timesteps = total_timesteps
        
    def _on_training_start(self):
        self.pbar = tqdm(total=self.total_timesteps)

    def _on_step(self):
        self.pbar.update(1)
        return True
    
    def _on_training_end(self):
        self.pbar.close()
        self.pbar = None

# Create environment for training
env = gym.make('Eplus-discrete-hot-v1')
env.seed(seed)
env.action_space.seed(seed)
env.observation_space.seed(seed)

# Create environment for evaluation
eval_env = gym.make('Eplus-discrete-hot-v1')
eval_env.seed(seed)
eval_env.action_space.seed(seed)
eval_env.observation_space.seed(seed)

# Create model:
model = DQN("MlpPolicy", env, seed=seed, verbose=1, 
        gamma = gamma,
        learning_starts = learning_starts,
        target_update_interval = target_update_interval,
        buffer_size = buffer_size,
        exploration_fraction = exploration_fraction,
        exploration_initial_eps = exploration_initial_eps,
        exploration_final_eps = exploration_final_eps)

# Create the evaluation callback
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./logs/",
    log_path="./logs/",
    eval_freq=int(eval_freq),
    n_eval_episodes=1,
    deterministic=True,
    render=False
)

# Create the progress bar callback
progress_callback = ProgressBarCallback(total_timesteps=total_timesteps)

# Combine callbacks
from stable_baselines3.common.callbacks import CallbackList
callbacks = CallbackList([progress_callback, eval_callback])

# Train model:
start_time = time.time()

model.learn(total_timesteps=total_timesteps, callback=callbacks)

print()
print("--- %s seconds taken to train ---" % (time.time() - start_time))

#model.save("DQN models/regular_DQN_hot")

env.close()
eval_env.close()

/usr/local/lib/python3.7/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:50: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  "You provided an OpenAI Gym environment. "
/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:175: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  "Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator."
/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:188: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:196: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  f"The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `{type(result)}`"


  0%|          | 0/245280 [00:00<?, ?it/s]

/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:220: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  "Core environment is written in old step API which returns one bool instead of two. "
/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:142: UserWarning: WARN: The obs returned by the `step()` method was expecting numpy array dtype to be float32, actual type: float64
  f"{pre} was expecting numpy array dtype to be {observation_space.dtype}, actual type: {obs.dtype}"
/usr/local/lib/python3.7/dist-packages/gym/utils/passive_env_checker.py:165: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/usr/local/lib/python3.7/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is n

Eval num_timesteps=17520, episode_reward=-34249.97 +/- 0.00
Episode length: 35040.00 +/- 0.00
-----------------------------------
| eval/               |           |
|    mean_ep_length   | 3.5e+04   |
|    mean_reward      | -3.42e+04 |
| rollout/            |           |
|    exploration_rate | 0.915     |
| time/               |           |
|    total_timesteps  | 17520     |
-----------------------------------
New best mean reward!


## Testing no DA

In [ ]:
# Test model for 1 year:

model = DQN.load("DQN models/regular_DQN_hot")

env = gym.make('Eplus-discrete-hot-v1')
env.seed(seed)
env.action_space.seed(seed)
env.observation_space.seed(seed)


start_time = time.time()

for i in range(1):
    obs = env.reset()

    rewards = []
    total_power = []
    in_temp = []
    out_temp = []
    actions = []
    comfort_penalties = []
    done = False
    current_month = 0

    while not done:
        # Predict next step:
        a, _ = model.predict(obs)
        # Observe next state and reward:
        obs, reward, done, info = env.step(int(a))

        # Store information for plotting:
        rewards.append(reward)
        total_power.append(info['total_power'])
        in_temp.append(info['temperature'])
        out_temp.append(info['out_temperature'])
        actions.append(int(a))
        comfort_penalties.append(info['comfort_penalty'])

        if info['month'] != current_month: # display results every month
            current_month = info['month']
            print('Reward: ', sum(rewards), info)
    print('Episode ', i, 'Mean reward: ', np.mean(rewards), 'Cumulative reward: ', sum(rewards))
env.close()



print()
print("--- %s seconds taken to run ---" % (time.time() - start_time))

In [ ]:
fig, axes = plt.subplots(nrows = 3, ncols = 2, figsize = (12,12))

axes[0,0].plot(in_temp)
axes[0,0].set_title('Indoor temperature ($^\circ$C)')
axes[0,0].set_xlabel('timestep')
axes[0,0].set_ylabel('temperature')

axes[0,1].plot(out_temp)
axes[0,1].set_title('Outdoor temperature ($^\circ$C)')
axes[0,1].set_xlabel('timestep')
axes[0,1].set_ylabel('temperature')

axes[1,0].plot(total_power)
axes[1,0].set_title('Power usage')
axes[1,0].set_xlabel('timestep')
axes[1,0].set_ylabel('units')

axes[1,1].plot(rewards)
axes[1,1].set_title('Reward')
axes[1,1].set_xlabel('timestep')
axes[1,1].set_ylabel('units')

axes[2,0].hist(actions, rwidth = 0.5)
axes[2,0].set_title('Actions taken')
axes[2,0].set_xlabel('action')
axes[2,0].set_ylabel('frequency')

axes[2,1].plot(comfort_penalties)
axes[2,1].set_title('Comfort penalty each timestep')
axes[2,1].set_xlabel('timestep')
axes[2,1].set_ylabel('comfort penalty ($^\circ$C)')


plt.tight_layout()


plt.savefig('project_images/DQN_metrics_hot.png')

In [ ]:
winter = actions[33700:] + actions[:7450]
spring = actions[7450:16150]
summer = actions[16150:24900]
autumn = actions[24900:33700]

len(winter), len(spring), len(summer), len(autumn)

In [ ]:
# split actions into seasons

fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (12,8))

axes[0,0].hist(winter, rwidth = 0.5)
axes[0,0].set_title('Winter')
axes[0,0].set_xlabel('action')
axes[0,0].set_ylabel('frequency')

axes[0,1].hist(spring, rwidth = 0.5)
axes[0,1].set_title('Spring')
axes[0,1].set_xlabel('action')
axes[0,1].set_ylabel('frequency')

axes[1,0].hist(summer, rwidth = 0.5)
axes[1,0].set_title('Summer')
axes[1,0].set_xlabel('action')
axes[1,0].set_ylabel('frequency')

axes[1,1].hist(autumn, rwidth = 0.5)
axes[1,1].set_title('Autumn')
axes[1,1].set_xlabel('action')
axes[1,1].set_ylabel('frequency')


plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5))

ax.hist([winter, spring, summer, autumn], rwidth = 0.8, 
            label = ['winter', 'spring', 'summer', 'autumn'], bins = np.arange(11) - 0.5)

ax.set_title('Actions by season')
ax.set_xlabel('action')
ax.set_ylabel('frequency')

ax.legend()

ax.set_ylim([0,5300])

#ax.set(xticks=range(10), xlim=[-1, 10])

ax.set_xticks(list(range(10)))


plt.savefig('project_images/DQN_actions_by_season_hot.png')

In [ ]:
print("Total comfort penalty:", sum(comfort_penalties))
print()
print("Number of comfort penalties:", sum([0 if (x == 0) else 1 for x in comfort_penalties]))

In [ ]:
# See how the actions vary during the day (7am-7pm) / night (7pm-7am):



day_time_steps = [(i*96 + j) for i in range(0, 365) for j in range(28,76)]

night_time_steps = [(i*96 + j) for i in range(0, 364) for j in range(76,124)]



day_actions = [actions[day_time_steps[i]] for i in range(len(day_time_steps))]

night_actions = [actions[night_time_steps[i]] for i in range(len(night_time_steps))]

fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,5))

ax.hist([day_actions, night_actions], rwidth = 0.8, 
            label = ['daytime', 'nighttime'], bins = np.arange(11) - 0.5)

ax.set_title('Actions by time of day')
ax.set_xlabel('action')
ax.set_ylabel('frequency')

ax.legend()

ax.set_xticks(list(range(10)))


plt.savefig('project_images/DQN_actions_by_time_hot.png')

# WITH DA

In [ ]:
########  DQN run with DA (hot environment) ########

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback

class DataAssimilationCallback(BaseCallback):
    
    def __init__(self, check_freq, actual_temp = None, actual_power = None, 
                  temp_DA = False, power_DA = False, both_DA = False, verbose=1):
        super(DataAssimilationCallback, self).__init__(verbose)
        self.check_freq = check_freq
        # Number of time the callback was called:
        self.n_calls = 0
        # Actual states after step:
        self.actual_temp = actual_temp
        self.actual_power = actual_power

        self.count = 0

        self.temp_DA = temp_DA
        self.power_DA = power_DA
        self.both_DA = both_DA



    def _on_step(self) -> bool:
        if self.n_calls % self.check_freq == 0:

          if self.temp_DA:
            env.update_temp(self.actual_temp[self.count])

          elif self.power_DA:
            env.update_power(self.actual_power[self.count])

          elif self.both_DA:
            env.update_both(self.actual_temp[self.count], self.actual_power[self.count])

          self.count += 4
          

        return True    # set false for just one step

In [ ]:
df = pd.read_csv("mean_runs/hot_means.csv")

df.head()

In [ ]:
new_temps = df['indoor_temp_mean'].tolist()

new_power = df['total_power_mean'].tolist()

new_temps[:5], new_power[:5]

In [ ]:
###### Temp DA only ######


# Create environment:
env = gym.make('Eplus-discrete-hot-v1')

# Create model:
model = DQN("MlpPolicy", env, verbose=1, 
        gamma = gamma,
        learning_starts = learning_starts,
        target_update_interval = target_update_interval,
        buffer_size = buffer_size,
        exploration_fraction = exploration_fraction,
        exploration_initial_eps = exploration_initial_eps,
        exploration_final_eps = exploration_final_eps)


# Callback:
callback = DataAssimilationCallback(check_freq = frequency, actual_temp = years*new_temps, temp_DA = True)

## Training

In [ ]:
start_time = time.time()

model.learn(total_timesteps=total_timesteps, 
            eval_freq = eval_freq, 
            n_eval_episodes = 1, 
            callback=callback)

print()
print("--- %s seconds taken to train ---" % (time.time() - start_time))


#model.save("DQN models/temp_DA_DQN_hot")

env.close()

## Testing

In [ ]:
# Test model for 1 year:

model = DQN.load("DQN models/temp_DA_DQN_hot")  

env = gym.make('Eplus-discrete-hot-v1')


start_time = time.time()

for i in range(1):
    obs = env.reset()

    rewards = []
    total_power = []
    in_temp = []
    out_temp = []
    actions = []
    comfort_penalties = []
    done = False
    current_month = 0

    while not done:
        # Predict next step:
        a, _ = model.predict(obs)
        # Observe next state and reward:
        obs, reward, done, info = env.step(int(a))

        # Store information for plotting:
        rewards.append(reward)
        total_power.append(info['total_power'])
        in_temp.append(info['temperature'])
        out_temp.append(info['out_temperature'])
        actions.append(int(a))
        comfort_penalties.append(info['comfort_penalty'])

        if info['month'] != current_month: # display results every month
            current_month = info['month']
            print('Reward: ', sum(rewards), info)
    print('Episode ', i, 'Mean reward: ', np.mean(rewards), 'Cumulative reward: ', sum(rewards))
env.close()


print()
print("--- %s seconds taken to run ---" % (time.time() - start_time))

In [ ]:
fig, axes = plt.subplots(nrows = 3, ncols = 2, figsize = (12,12))

axes[0,0].plot(in_temp)
axes[0,0].set_title('Indoor temperature ($^\circ$C)')
axes[0,0].set_xlabel('timestep')
axes[0,0].set_ylabel('temperature')

axes[0,1].plot(out_temp)
axes[0,1].set_title('Outdoor temperature ($^\circ$C)')
axes[0,1].set_xlabel('timestep')
axes[0,1].set_ylabel('temperature')

axes[1,0].plot(total_power)
axes[1,0].set_title('Power usage')
axes[1,0].set_xlabel('timestep')
axes[1,0].set_ylabel('units')

axes[1,1].plot(rewards)
axes[1,1].set_title('Reward')
axes[1,1].set_xlabel('timestep')
axes[1,1].set_ylabel('units')

axes[2,0].hist(actions, rwidth = 0.5)
axes[2,0].set_title('Actions taken')
axes[2,0].set_xlabel('action')
axes[2,0].set_ylabel('frequency')

axes[2,1].plot(comfort_penalties)
axes[2,1].set_title('Comfort penalty each timestep')
axes[2,1].set_xlabel('timestep')
axes[2,1].set_ylabel('comfort penalty ($^\circ$C)')


plt.tight_layout()


plt.savefig('project_images/DA_DQN_metrics_hot.png')

In [ ]:
winter = actions[33700:] + actions[:7450]
spring = actions[7450:16150]
summer = actions[16150:24900]
autumn = actions[24900:33700]

len(winter), len(spring), len(summer), len(autumn)

In [ ]:
# split actions into seasons

fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (12,8))

axes[0,0].hist(actions[34000:] + actions[:7450], rwidth = 0.5)
axes[0,0].set_title('Winter')
axes[0,0].set_xlabel('action')
axes[0,0].set_ylabel('frequency')

axes[0,1].hist(actions[7450:16150], rwidth = 0.5)
axes[0,1].set_title('Spring')
axes[0,1].set_xlabel('action')
axes[0,1].set_ylabel('frequency')

axes[1,0].hist(actions[16150:24900], rwidth = 0.5)
axes[1,0].set_title('Summer')
axes[1,0].set_xlabel('action')
axes[1,0].set_ylabel('frequency')

axes[1,1].hist(actions[24900:34000], rwidth = 0.5)
axes[1,1].set_title('Autumn')
axes[1,1].set_xlabel('action')
axes[1,1].set_ylabel('frequency')


plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(nrows =1, ncols = 1, figsize = (10,5))

ax.hist([winter, spring, summer, autumn], rwidth = 0.8, 
            label = ['winter', 'spring', 'summer', 'autumn'], bins = np.arange(11) - 0.5)

ax.set_title('Actions by season')
ax.set_xlabel('action')
ax.set_ylabel('frequency')

ax.legend()

ax.set_ylim([0,5000])

ax.set_xticks(list(range(10)))


plt.savefig('project_images/DA_DQN_actions_by_season_hot.png')

In [ ]:
print("Total comfort penalty:", sum(comfort_penalties))
print()
print("Number of comfort penalties:", sum([0 if (x == 0) else 1 for x in comfort_penalties]))